In [1]:
from rich.console import Console
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv(override=True)

True

In [2]:
def show(text: str):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
show("Hello World!")

Hello World!

In [4]:
openai = OpenAI()

In [5]:
todos = []
completed = []

In [6]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"

    show(result)
    return result

In [7]:
get_todo_report()

''

In [8]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))

    return get_todo_report()

In [9]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index"
    
    show(completion_notes)
    return get_todo_report()

In [10]:
todos, completed = [], []
create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [12]:
create_todos_json={
    "name": "create_todos",
    "description": "Add new todos from the list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "title": "Descriptions"
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [13]:
mark_complete_json={
    "name": "mark_complete",
    "description": "Mark complete a todo at a given position (start with 1) and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "description": "The 1-based index of the todo to mark as complete",
                "title": "Index",
                "type": "integer"
            },
            "completion_notes": {
                "description": "Notes about how you competed the todo in rich console markup",
                "title": "Completion Notes",
                "type": "string"
            }
        },
        "reuquired": ["index", "completion_notes"],
        "type": "object",
        "additionalProperties": False
    }
}

In [14]:
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json}
]

In [15]:
tools

[{'type': 'function',
  'function': {'name': 'create_todos',
   'description': 'Add new todos from the list of descriptions and return the full list',
   'parameters': {'type': 'object',
    'properties': {'descriptions': {'type': 'array',
      'items': {'type': 'string'},
      'title': 'Descriptions'}},
    'required': ['descriptions'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'mark_complete',
   'description': 'Mark complete a todo at a given position (start with 1) and return the full list',
   'parameters': {'type': 'object',
    'properties': {'index': {'description': 'The 1-based index of the todo to mark as complete',
      'title': 'Index',
      'type': 'integer'},
     'completion_notes': {'description': 'Notes about how you competed the todo in rich console markup',
      'title': 'Completion Notes',
      'type': 'string'}},
    'reuquired': ['index', 'completion_notes'],
    'additionalProperties': False}}}]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})

    return results

In [25]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls=tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True

    show(response.choices[0].message.content)


In [22]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each jstep in turn.
Now use the todo list steps, create a plan, carry out the steps, and reply with a solution.
If any quantity is not provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in a rich console markup without code block.
Do not ask the user with questions or clarification; respond only with the answer after using your tools.
"""

user_message = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

In [23]:
messages

[{'role': 'system',
  'content': '\nYou are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each jstep in turn.\nNow use the todo list steps, create a plan, carry out the steps, and reply with a solution.\nIf any quantity is not provided in the question, then include a step to come up with a reasonable estimate.\nProvide your solution in a rich console markup without code block.\nDo not ask the user with questions or clarification; respond only with the answer after using your tools.\n'},
 {'role': 'user',
  'content': '\nA train leaves Boston at 2:00 pm traveling 60 mph.\nAnother train leaves New York at 3:00 pm traveling 80 mph toward Boston.\nWhen do they meet?\n'}]

In [26]:
todos, completed = [], []
loop(messages=messages)

Todo #1: Identify initial time offset between departures and compute head-start distance for the Boston train by 
3:00 pm.
Todo #2: Let t be time after 3:00 pm until meeting; set up relative motion equation using combined speed and 
remaining distance.
Todo #3: If total distance between Boston and New York is not given, estimate a reasonable value and state it; then
solve for meeting time.
Todo #4: Compute clock time and present result clearly.

Boston train departs at 2:00 pm at 60 mph. By 3:00 pm it has traveled 1 hour × 60 mph = 60 miles toward New York.

Todo #1: Identify initial time offset between departures and compute head-start distance for the Boston train by 
3:00 pm.
Todo #2: Let t be time after 3:00 pm until meeting; set up relative motion equation using combined speed and 
remaining distance.
Todo #3: If total distance between Boston and New York is not given, estimate a reasonable value and state it; then
solve for meeting time.
Todo #4: Compute clock time and present result clearly.

After 3:00 pm, the trains approach each other with relative speed 60 + 80 = 140 mph. If D is the Boston–New York 
distance, then at 3:00 pm the remaining separation is (D − 60), so meeting time t (hours after 3:00) satisfies 140 
t = D − 60.

Todo #1: Identify initial time offset between departures and compute head-start distance for the Boston train by 
3:00 pm.
Todo #2: Let t be time after 3:00 pm until meeting; set up relative motion equation using combined speed and 
remaining distance.
Todo #3: If total distance between Boston and New York is not given, estimate a reasonable value and state it; then
solve for meeting time.
Todo #4: Compute clock time and present result clearly.

The problem omits D, so we must estimate. A common estimate for Boston–NYC distance is about 215 miles (approximate
driving distance). Using D = 215: 140 t = 215 − 60 = 155 ⇒ t = 155/140 = 1.1071 hr ≈ 1 hr 6 min 26 sec.

Todo #1: Identify initial time offset between departures and compute head-start distance for the Boston train by 
3:00 pm.
Todo #2: Let t be time after 3:00 pm until meeting; set up relative motion equation using combined speed and 
remaining distance.
Todo #3: If total distance between Boston and New York is not given, estimate a reasonable value and state it; then
solve for meeting time.
Todo #4: Compute clock time and present result clearly.

Add t ≈ 1:06:26 to 3:00 pm → about 4:06 pm (≈ 4:06:26 pm).

Todo #1: Identify initial time offset between departures and compute head-start distance for the Boston train by 
3:00 pm.
Todo #2: Let t be time after 3:00 pm until meeting; set up relative motion equation using combined speed and 
remaining distance.
Todo #3: If total distance between Boston and New York is not given, estimate a reasonable value and state it; then
solve for meeting time.
Todo #4: Compute clock time and present result clearly.

Setup  
• By 3:00 pm, the Boston train (60 mph) has a 1-hour head start ⇒ it has gone 60 miles.  
• After 3:00 pm, they travel toward each other with combined (relative) speed:  
 60 + 80 = 140 mph.

Distance needed  
The Boston–New York distance isn’t given, so use a reasonable estimate: ~215 miles (about the driving distance).

Solve  
At 3:00 pm, remaining separation ≈ 215 − 60 = 155 miles.  
Time after 3:00 pm to meet:  
 t = 155 / 140 = 1.1071 hours ≈ 1 hour 6 minutes.

Meeting time  
3:00 pm + 1:06 ≈ 4:06 pm (about 4:06:26 pm).